## Handoff Pattern

In [1]:
import os
from typing import Callable, Literal

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

In [4]:
AgentName = Literal["triage", "billing", "technical"]


# --- State -------------------------------------------------------------------
class SupportState(AgentState):
    """Conversation state; `active_agent` persists across turns."""

    active_agent: AgentName


In [5]:
# --- Domain tools (only visible to the agent that owns them) -----------------
@tool
def lookup_invoice(invoice_id: str) -> str:
    """Look up an invoice by its ID."""
    return f"Invoice {invoice_id}: $120.00, status=PAID, charged twice on 2026-10-01."


@tool
def issue_refund(invoice_id: str, amount: float) -> str:
    """Issue a refund for an invoice."""
    return f"Refund of ${amount:.2f} issued for invoice {invoice_id}."


@tool
def check_service_status(service: str) -> str:
    """Check the current status of a service."""
    return f"Service '{service}' is operational. Last incident: none in 30 days."


@tool
def restart_device(device_id: str) -> str:
    """Remotely restart a customer's device."""
    return f"Restart command sent to device {device_id}."


In [6]:
# --- Handoff tools -----------------------------------------------------------
def _handoff(runtime: ToolRuntime, target: AgentName, message: str) -> Command:
    # The ToolMessage matching the tool_call_id is required, otherwise the
    # message history is malformed (an AI tool call with no response).
    return Command(
        update={
            "messages": [ToolMessage(content=message, tool_call_id=runtime.tool_call_id)],
            "active_agent": target,
        }
    )

@tool
def transfer_to_billing(runtime: ToolRuntime) -> Command:
    """Hand off to the billing specialist (invoices, charges, refunds)."""
    return _handoff(runtime, "billing", "Transferred to billing specialist.")


@tool
def transfer_to_technical(runtime: ToolRuntime) -> Command:
    """Hand off to the technical support specialist (outages, devices, errors)."""
    return _handoff(runtime, "technical", "Transferred to technical support specialist.")


@tool
def transfer_to_triage(runtime: ToolRuntime) -> Command:
    """Hand back to triage when the issue is resolved or is outside your scope."""
    return _handoff(runtime, "triage", "Transferred back to triage.")



In [7]:
# --- Per-agent configuration -------------------------------------------------
AGENTS: dict[str, dict] = {
    "triage": {
        "prompt": (
            "You are a friendly customer support triage agent. Work out what the "
            "customer needs and hand off to the right specialist: billing for "
            "invoices/charges/refunds, technical for outages/devices/errors. "
            "Do not try to solve specialist problems yourself."
        ),
        "tools": [transfer_to_billing, transfer_to_technical],
    },
    "billing": {
        "prompt": (
            "You are a billing specialist. Look up invoices, and ask the customer to "
            "confirm before issuing any refund. Reply to the customer after each step. "
            "Only hand back to triage when the customer raises a non-billing issue "
            "or says they are done."
        ),
        "tools": [lookup_invoice, issue_refund, transfer_to_triage],
    },
    "technical": {
        "prompt": (
            "You are a technical support specialist. Check service status, and ask "
            "the customer to confirm before restarting a device. Reply to the customer "
            "after each step. Only hand back to triage when the customer raises a "
            "non-technical issue or says they are done."
        ),
        "tools": [check_service_status, restart_device, transfer_to_triage],
    },
}


In [8]:
ALL_TOOLS = [
    lookup_invoice,
    issue_refund,
    check_service_status,
    restart_device,
    transfer_to_billing,
    transfer_to_technical,
    transfer_to_triage,
]


In [9]:
# --- Middleware: swap prompt + tools based on active_agent -------------------
@wrap_model_call
def apply_active_agent(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    active = request.state.get("active_agent", "triage")
    config = AGENTS[active]
    return handler(
        request.override(
            system_prompt=config["prompt"],
            tools=config["tools"],
        )
    )


In [10]:
agent = create_agent(
    model,
    tools=ALL_TOOLS,  # must register every tool so ToolNode can execute any of them
    state_schema=SupportState,
    middleware=[apply_active_agent],
    checkpointer=InMemorySaver(),
)


In [11]:
# --- Demo --------------------------------------------------------------------
def chat(thread_id: str, text: str) -> None:
    print(f"\nUser: {text}")
    result = agent.invoke(
        {"messages": [HumanMessage(text)]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"[{result['active_agent']}] {result['messages'][-1].content}")


In [12]:
if __name__ == "__main__":
    thread = "demo-1"
    chat(thread, "Hi, I was charged twice for invoice INV-1001.")  # triage -> billing
    chat(thread, "Yes please refund the duplicate $120.")  # billing handles it
    chat(thread, "Also, my router R-77 keeps dropping the connection.")  # -> triage -> technical



User: Hi, I was charged twice for invoice INV-1001.
[billing] I found invoice **INV-1001** for **$120.00**, and it shows two charges on **October 1, 2026**. Would you like me to issue a **$120.00 refund** for the duplicate charge?

User: Yes please refund the duplicate $120.
[billing] A **$120.00 refund** for the duplicate charge on invoice **INV-1001** has been issued successfully.

User: Also, my router R-77 keeps dropping the connection.
[technical] The internet service is currently operational, with no incidents reported in the past 30 days. Your router **R-77** may need a restart. Would you like me to restart it remotely?
